In [1]:
import pandas as pd
import duckdb

arquivo = "yellow_tripdata_2026-01.parquet"
df_duckdb = pd.read_parquet(arquivo)
con = duckdb.connect()
con.execute(f"""
   CREATE TABLE corridas AS
   SELECT * FROM read_parquet('{arquivo}')
""")

# Inspecionar o número de linhas, as colunas e os tipos de dados
con.table("corridas").show()

df_duckdb = con.table("corridas").df()
df_duckdb.info()

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┬────────────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ Airport_fee │ cbd_congestion_fee │
│  int32   │      timestamp       │       timestamp       │      int64      │    double     │   int64    │      varchar       │    int32     │    int32     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │       double       │
├──────

### Questão 1

**Corridas por horário (sem IA):** Para cada hora de início da corrida (0 a 23, a partir de
tpep_pickup_datetime), calcule o número de corridas, a distância média (trip_distance)
e o valor médio pago (total_amount). Ordene pelo número de corridas, do maior para o
menor. Utilize os registros de janeiro de 2026.

In [2]:
# DuckDB
df_duckdb_hora_inicio = con.execute(f"""
    SELECT 
        HOUR(tpep_pickup_datetime) AS hora_inicio_corrida,
        COUNT(*) AS num_corridas,
        AVG(trip_distance) AS distancia_media,
        AVG(total_amount) AS valor_medio_pago
    FROM read_parquet('{arquivo}')
    WHERE tpep_pickup_datetime BETWEEN '2026-01-01' AND '2026-02-01'
    GROUP BY hora_inicio_corrida
    ORDER BY num_corridas DESC
""").df()

print(df_duckdb_hora_inicio)

    hora_inicio_corrida  num_corridas  distancia_media  valor_medio_pago
0                    18        265574         3.191099         29.317995
1                    17        252173         6.149070         30.089539
2                    19        234542         4.336286         29.063146
3                    16        227893         4.936715         30.851314
4                    15        221677         5.255241         29.340657
5                    20        212883         5.726825         29.116929
6                    21        212544         7.311580         29.230308
7                    14        207507         6.578414         29.071705
8                    13        195708         5.526828         27.888121
9                    22        194801         5.441168         29.510664
10                   12        187435         5.295283         27.497725
11                   11        171729         7.003858         27.032998
12                   10        161419         9.127

In [3]:
# Pandas
df_pandas = pd.read_parquet(arquivo)

df_pandas['tpep_pickup_datetime'] = pd.to_datetime(df_pandas['tpep_pickup_datetime'])

df_filtered = df_pandas.query("'2026-01-01' <= tpep_pickup_datetime <= '2026-02-01'")

df_pandas_hora_inicio = (
    df_filtered.groupby(df_filtered['tpep_pickup_datetime'].dt.hour).agg(
                                                                        num_corridas=('tpep_pickup_datetime', 'count'),
                                                                        distancia_media=('trip_distance', 'mean'),
                                                                        valor_medio_pago=('total_amount', 'mean'),
                                                                    ).reset_index(names='hora_inicio_corrida').sort_values('num_corridas', ascending=False)
)

print(df_pandas_hora_inicio)

    hora_inicio_corrida  num_corridas  distancia_media  valor_medio_pago
18                   18        265574         3.191099         29.317995
17                   17        252173         6.149070         30.089539
19                   19        234542         4.336286         29.063146
16                   16        227893         4.936715         30.851314
15                   15        221677         5.255241         29.340657
20                   20        212883         5.726825         29.116929
21                   21        212544         7.311580         29.230308
14                   14        207507         6.578414         29.071705
13                   13        195708         5.526828         27.888121
22                   22        194801         5.441168         29.510664
12                   12        187435         5.295283         27.497725
11                   11        171729         7.003858         27.032998
10                   10        161419         9.127